**4**

Name your Jupyter Notebook as:

`TASK4_<your name>_<centre number>_<index number>.ipynb`

A holiday company has 10 villas that customers can hire. A program is being created to store the customer bookings and allow the company staff to check the availability of the villas on requested dates.

The text files `customerBookings.txt` and `villas.txt` store the current data.

The data in `villas.txt` is stored in the following format:

`villa ID, villa name, country, cost`

The data in `customerBookings.txt` is stored in the following format:

`booking ID, customer ID, villa ID, start date, number of days`

You do not need to consider how data about the customers will be stored.

You can assume the year is 2025.

For each of the sub-tasks, add a comment statement at the beginning of the code, using the hash symbol '#', to indicate the sub-task the program code belongs to, for example:

```
In [1]: #Task 4.1
        Program code

        Output:
```


**Task 4.1**

Write a Python program that uses SQL to create a database with two tables: one table to store data about the villas and one table to store data about customer bookings.

Define the primary and foreign keys for each table. **[7]**


In [2]:
#Task 4.1
import sqlite3
conn = sqlite3.connect("Holiday.db")

conn.execute("DROP TABLE IF EXISTS Villa_Booking")
conn.execute("DROP TABLE IF EXISTS Booking")
conn.execute("DROP TABLE IF EXISTS Villa")

conn.execute("""
            CREATE TABLE Villa(
            villaID INTEGER,
            villaName TEXT NOT NULL,
            country TEXT NOT NULL,
            cost TEXT NOT NULL,
            PRIMARY KEY(villaID))
            """)

conn.execute("CREATE TABLE Booking("
             "bookingID INTEGER,"
             "customerID TEXT NOT NULL,"
             "villaID TEXT NOT NULL,"
             "startdate TEXT NOT NULL,"
             "numberofdays INTEGER NOT NULL,"
             "FOREIGN KEY(villaID) REFERENCES Villa(villaID),"
             "PRIMARY KEY(bookingID))")

conn.commit()


**Task 4.2**

Write a Python program that uses SQL to insert the data from each of the two text files into the appropriate field in each database table. **[4]**

Run your program to test the data has been entered correctly. **[1]**


In [3]:
#Task 4.2
with open("villas.txt", 'r') as file:
    data = []
    for i in file:
        data.append(i)
    #file automatically closes
for i in data:
    i = i.strip('\n').split(',')
    conn.execute("INSERT INTO Villa(villaID, villaName, country, cost) VALUES(?,?,?,?)", (i[0], i[1], i[2], i[3],))
conn.commit()

with open("customerBookings.txt", 'r') as file1:
    data1 = []
    for i in file1:
        data1.append(i)
    #file automatically closes
for i in data1:
    i = i.strip('\n').split(',')
    conn.execute("INSERT INTO Booking(bookingID, customerID, villaID, startdate, numberofdays) VALUES(?,?,?,?,?)", (i[0], i[1], i[2], i[3], i[4],))
conn.commit()
conn.close()

**Task 4.3**

A new table `Villa_Booking`, needs creating to store every date that each villa has been booked. This will have two fields: one for the villa ID and one for the date.

For example, if villa number 1 is booked from the 01-Jan for 4 days, the following data will be inserted into `Villa_Booking`:

```
1   01-Jan
1   02-Jan
1   03-Jan
1   04-Jan
```

Write a Python program that uses SQL to create `Villa_Booking` and populate it with the existing customer bookings. **[5]**


In [4]:
#Task 4.3
conn = sqlite3.connect("Holiday.db")

data = conn.execute("SELECT Booking.villaID, startdate, numberofdays FROM Villa, Booking WHERE Villa.villaID = Booking.villaID").fetchall()

conn.execute("CREATE TABLE Villa_Booking("
            "villaID INTEGER,"
            "date TEXT NOT NULL,"
            "FOREIGN KEY(villaID) REFERENCES Villa(villaID),"
            "PRIMARY KEY(villaID, date))")

for i in data:
    villa, start, number = i
    for i in range(number):
        conn.execute("INSERT INTO Villa_Booking(villaID, date) VALUES(?,?)", (villa, start))
        nextday = str(int(start[:2]) + 1)
        if len(nextday) == 1:
            start = f"0{nextday}{start[2:]}"
        else:
            start = f"{nextday}{start[2:]}"
conn.commit()
conn.close()
        
    

**Task 4.4**

Write a Python function to:

- take as input the villa name, start date and number of days a customer would like to book
- use SQL to check the availability of the villa for the customer's request
- output a list of all the requested dates that the villa is available
- output a list of all the requested dates that the villa is not available.

All outputs must have an appropriate message. **[6]**

Test your program with the following data:

```
Villa name       Dolphin
Month            Apr
Date             8
Number of days   4
```

**[1]**

Save your Jupyter Notebook for Task 4.


In [12]:
#Task 4.4
def check():
    conn = sqlite3.connect("Holiday.db")
    name = input("Villa Name: ")
    month = input("Month: ")
    date = input("Date: ")
    duration = int(input("Number of days: "))
    month = '-' + month
    if len(date) == 1:
        start = f"0{date}{month}"
    else:
        start = f"{date}{month}"
    data = conn.execute("SELECT villaID FROM Villa WHERE villaName = ?",(name,)).fetchone()
    data = data[0]
    no = []
    yes = []
    for i in range(duration):
        available = conn.execute("SELECT * FROM Villa_Booking WHERE villaID = ? and date = ?",(data, start,)).fetchall()
        if available == []:
            no.append(start)
            nextday = str(int(start[:2]) + 1)
            if len(nextday) == 1:
                start = f"0{nextday}{start[2:]}"
            else:
                start = f"{nextday}{start[2:]}"
        else:
            yes.append(start)
            nextday = str(int(start[:2]) + 1)
            if len(nextday) == 1:
                start = f"0{nextday}{start[2:]}"
            else:
                start = f"{nextday}{start[2:]}"
    print(name, "is not available on:")
    print(no)
    print(name, "is available on:")
    print(yes)
    return

check()

Villa Name: Dolphin
Month: Apr
Date: 8
Number of days: 4
[]
[]
[(3, '10-Apr')]
[(3, '11-Apr')]
Dolphin is not available on:
['08-Apr', '09-Apr']
Dolphin is available on:
['10-Apr', '11-Apr']
